### Prof. Alfio Ferrara

# La memoria della macchina

##  Metodologie informatiche nelle discipline umanistiche
##### A.A. 2026/2027

---

Nel notebook precedente abbiamo dettato istruzioni alla tartaruga e abbiamo visto un interprete trasformare un testo in un disegno. Ora ci chiediamo cosa la macchina tiene con sé mentre lavora, e come possiamo dirglielo.

In [32]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Che cosa ricorda la tartaruga

Ripartiamo dalla tartaruga. Le diamo un programma che cambia più cose insieme (colore, spessore, posizione, direzione, penna) e guardiamo tutto quello che la tartaruga conserva prima e dopo l'esecuzione.

In [33]:
from nlp.toy_language import Turtle, tokenize, parse, execute

program = """
colore rosso
spessore 4
avanti 100
destra 90
alza
avanti 50
"""

names = ["x", "y", "heading", "pen_down", "color", "width"]
turtle = Turtle()
before = {name: getattr(turtle, name) for name in names}
execute(parse(tokenize(program)), turtle)

def show_value(value):
    return round(value, 1) if isinstance(value, float) else value

print(f"{'nome':<12}{'prima':>10}{'dopo':>10}")
for name in names:
    print(f"{name:<12}{show_value(before[name])!s:>10}{show_value(getattr(turtle, name))!s:>10}")

nome             prima      dopo
x                  0.0      50.0
y                  0.0     100.0
heading           90.0       0.0
pen_down          True     False
color            black       red
width              2.0       4.0


## Un momento di riflessione

- Ogni riga della tabella ha un nome e un valore. A cosa serve il nome? Cosa cambierebbe se la tartaruga tenesse solo una fila di numeri, senza etichette?
- Quali valori sono cambiati e quali no? Cosa li ha cambiati?
- Il valore di `x` dopo `avanti 50` dipende solo da quel comando, o anche da qualcosa che c'era già prima? Cosa succederebbe se la tartaruga dimenticasse tutto dopo ogni comando?
- Pensate a un'applicazione che usate ogni giorno: cosa deve ricordare tra un tocco sullo schermo e il successivo?

## Le variabili

Ciò che abbiamo appena guardato ha una forma precisa. Una **variabile** è un posto nella memoria della macchina con un **nome**, che contiene un **valore**. Il valore può cambiare nel tempo, il nome no: è quello che ci permette di ritrovarlo.

Le variabili servono a tre cose:

- **ricordare** un'informazione da un'istruzione alla successiva (dove sono, cosa ho già fatto);
- **dare un nome al significato**: `lato` dice più di `100`, e se cambia lo cambio in un punto solo;
- **far evolvere un programma**: lo stato di un programma in un momento è l'insieme dei valori delle sue variabili, e ogni istruzione lo modifica.

Nella tartaruga le variabili erano nascoste dentro l'interprete e le usavamo senza scriverle. Ora impariamo a crearle noi.

## I tipi

Torniamo alla tabella della tartaruga. I valori non sono tutti dello stesso genere: ci sono numeri, un sì/no, una parola. Python tiene conto di questa differenza e la chiama **tipo**. Per ogni nome, aggiungiamo alla tabella il tipo del valore.

In [34]:
from nlp.toy_language import Turtle

turtle = Turtle()
names = ["x", "y", "heading", "pen_down", "color", "width"]

print(f"{'nome':<12}{'valore':>10}   tipo")
for name in names:
    value = getattr(turtle, name)
    print(f"{name:<12}{value!s:>10}   {type(value).__name__}")

nome            valore   tipo
x                  0.0   float
y                  0.0   float
heading           90.0   float
pen_down          True   bool
color            black   str
width              2.0   float


In [ ]:
a = "5"
print(type(a))
a = a + "2x"
print(a)

a = 5
b = 5
c = a >= b
print(c)

<class 'str'>
52x
False


Il tipo non è un'etichetta decorativa: dice come leggere i bit. Ricordate `3` e `5` in L01, quando la loro somma dava `h`? Scriviamo **42** in quattro modi diversi e guardiamo cosa c'è in memoria.

In [48]:
import struct

def bits(data):
    return " ".join(format(byte, "08b") for byte in data)

rows = [
    (42, format(42, "08b")),
    ("42", bits("42".encode())),
    (42.0, bits(struct.pack(">d", 42.0))),
    (True, format(True, "b")),
]

print(f"{'valore':<8}{'tipo':<8}bit")
for value, representation in rows:
    print(f"{value!r:<8}{type(value).__name__:<8}{representation}")

valore  tipo    bit
42      int     00101010
'42'    str     00110100 00110010
42.0    float   01000000 01000101 00000000 00000000 00000000 00000000 00000000 00000000
True    bool    1


Se il tipo dice come leggere i bit, dice anche cosa si può fare con un valore. Per ognuna di queste espressioni chiediamoci cosa risponderà Python, e poi controlliamo.

In [49]:
questions = ["3 + 5", '"3" + "5"', '"3" * 3', '3 + "5"', "10 > 9", '"10" > "9"', "7 / 2", "7 // 2", "0.1 + 0.2"]

for expression in questions:
    print("Cosa risponde Python a:", expression)
    input("Invio per vedere la risposta")
    try:
        result = eval(expression)
        print(f"  {result!r}  ({type(result).__name__})")
    except TypeError as error:
        print(f"  errore, {type(error).__name__}: {error}")
    print()

Cosa risponde Python a: 3 + 5
  8  (int)

Cosa risponde Python a: "3" + "5"
  '35'  (str)

Cosa risponde Python a: "3" * 3
  '333'  (str)

Cosa risponde Python a: 3 + "5"
  errore, TypeError: unsupported operand type(s) for +: 'int' and 'str'

Cosa risponde Python a: 10 > 9
  True  (bool)

Cosa risponde Python a: "10" > "9"
  False  (bool)

Cosa risponde Python a: 7 / 2
  3.5  (float)

Cosa risponde Python a: 7 // 2
  3  (int)

Cosa risponde Python a: 0.1 + 0.2
  0.30000000000000004  (float)



Quando un valore ha il tipo sbagliato per quello che vogliamo fare, possiamo **convertirlo**: chiedere a Python di rileggerlo come un altro tipo. Vediamo alcune conversioni, comprese quelle che non vanno a buon fine.

In [ ]:
conversions = ['int("42")', "str(42)", 'float("3.5")', "int(3.9)", 'int("10110010", 2)', 'int("abc")', 'bool("")', 'bool("0")']

for expression in conversions:
    try:
        result = eval(expression)
        print(f"{expression:<22}{result!r:>8}  ({type(result).__name__})")
    except ValueError as error:
        print(f"{expression:<22}errore, {type(error).__name__}: {error}")

## Un momento di riflessione

- Il numero `42` e la stringa `"42"` si scrivono quasi allo stesso modo, ma in memoria sono bit diversi. Chi decide come leggerli: i bit stessi, o il tipo?
- `"3" + "5"` dà `"35"` e `3 + 5` dà `8`. Perché lo stesso `+` fa due cose diverse? E `3 + "5"` perché non fa nessuna delle due?
- `"10" > "9"` è falso. Come starebbero ordinati in un elenco i nomi di file `1.txt`, `2.txt`, `10.txt`? Cosa c'entrano i codici dei caratteri di L01?
- `int(3.9)` dà `3` senza avvisare. Una conversione può perdere informazione? Ricordate cosa succede quando si comprime un'immagine.
- Ricordate `0.1 + 0.2`, che avevamo già incontrato in L01. Cosa ha in comune con la scelta di un tipo?
- Nei dati di L02 (JSON, Wikidata) i valori avevano un tipo. Perché conta saperlo prima di usare un dato?

## Dare un nome a un valore

Fin qui i nomi li ha scelti l'interprete della tartaruga. Ora li scegliamo noi. Cominciamo cancellando tutto quello che Python ha in memoria in questo notebook, e chiedendogli cosa contiene.

In [ ]:
%reset -f
%whos

Per creare una variabile, in Python si usa il simbolo `=`: a sinistra il nome, a destra il valore.

In [ ]:
lato = 100
nome = "Dante"
%whos

Proviamo a usare `lato` per calcolare qualcos'altro, e poi a cambiarlo.

In [ ]:
perimetro = lato * 4
lato = 150
%whos

Ora una serie di piccoli programmi. Per ognuno chiediamoci quanto varranno le variabili alla fine, e poi controlliamo.

In [ ]:
programs = [
    "x = 3\nx = x + 1",
    "a = 5\nb = a\na = 10",
    "a = 5\nb = 10\na = b\nb = a",
    "n = 5\nn = \"cinque\"",
    "n = \"100\"\nn = n * 2",
    "lato = 100\nprint(Lato)",
    "y = y + 1",
]

for source in programs:
    print(source)
    print()
    input("Quanto valgono le variabili alla fine? Invio per vedere la risposta")
    namespace = {}
    try:
        exec(source, namespace)
        namespace.pop("__builtins__")
        for name, value in namespace.items():
            print(f"  {name} = {value!r}  ({type(value).__name__})")
    except NameError as error:
        print(f"  errore, {type(error).__name__}: {error}")
    print("-" * 40)

### Come si scelgono i nomi

Python accetta nomi fatti di lettere, cifre e il simbolo `_`, che non comincino con una cifra e non contengano spazi. Le maiuscole contano: `lato` e `Lato` sono nomi diversi, come abbiamo visto. Per Python un nome è solo un'etichetta, ma per chi legge il codice (voi tra una settimana, o un collega) è documentazione: `lato` dice qualcosa, `x` e `l1` molto meno.

## I valori che arrivano da fuori

Finora abbiamo scritto noi i valori nel codice. Spesso invece arrivano da fuori: da chi usa il programma, da un file, da un servizio in rete. La funzione `input()` chiede un valore a chi sta davanti alla tastiera.

In [ ]:
birth_year = input("In che anno sei nato? ")
print(repr(birth_year), type(birth_year).__name__)

Proviamo a calcolare l'età.

In [ ]:
try:
    print("Hai", 2026 - birth_year, "anni")
except TypeError as error:
    print(f"errore, {type(error).__name__}: {error}")

birth_year = int(birth_year)
print("Hai", 2026 - birth_year, "anni")

## Un momento di riflessione

- Nella cella con `perimetro` abbiamo cambiato `lato` dopo aver calcolato il perimetro, e `perimetro` vale ancora 400. Perché? Quando avviene il calcolo?
- In matematica `x = x + 1` non ha senso. In Python invece è una delle istruzioni più comuni. Cosa viene calcolato per primo, e cosa viene modificato?
- Nel tentativo di scambiare `a` e `b`, alla fine hanno lo stesso valore. Quale informazione è andata persa, e in quale momento? Come la si potrebbe conservare?
- `n = 5` e poi `n = "cinque"`: il tipo appartiene al nome o al valore? Come si conciliano le due cose con la tabella dei tipi della sezione precedente?
- `input()` restituisce sempre una stringa. Perché non può sapere se stiamo scrivendo un anno, un nome o un codice? Chi deve decidere il tipo, e dove lo dichiariamo nel codice?
- Se eseguissimo queste celle in un ordine diverso, o la stessa cella due volte, cosa ne sarebbe della memoria? Cosa succede alle variabili quando lo stato che abbiamo in testa e quello nel computer non coincidono?

## Il lago ghiacciato

Ora usiamo le variabili per costruire qualcosa. Il gioco si chiama *Frozen Lake* ed è ispirato a uno degli ambienti di prova della libreria Gymnasium, usata per far imparare giochi alle macchine.

Siete su un lago ghiacciato. Partite dalla casella verde in alto a sinistra e dovete raggiungere la bandiera in basso a destra. Le caselle blu sono ghiaccio solido, quelle nere sono buchi: chi ci finisce dentro cade nel lago e perde. Ci si muove scrivendo `w` (su), `a` (sinistra), `s` (giù), `d` (destra), seguiti da Invio. Con `q` si esce.

Giochiamo.

In [50]:
from nlp.frozen_lake import play

play()

🟩🟦🟦🟦
🟦⬛🟦⬛
🟦🟦🟦⬛
⬛🟦🟦🎉

Hai raggiunto il traguardo in 6 mosse!


Ora il ghiaccio è scivoloso: a volte vi porta dove non volevate andare.

In [ ]:
play(slippery=True)

## Un momento di riflessione

- Mentre giocavate, cosa doveva ricordare il gioco da una mossa alla successiva? Cosa cambia a ogni mossa e cosa non cambia mai?
- Per decidere cosa succede dopo la vostra mossa, cosa deve sapere il gioco? Da dove prende queste informazioni?
- Cosa succede se vi muovete contro il bordo del lago? Chi ha deciso questa regola? E cosa succede quando finite in un buco o sulla bandiera?
- Con il ghiaccio scivoloso, ripetendo la stessa sequenza di mosse si ottiene lo stesso risultato? Cosa cambia nel modo di descrivere il gioco?
- Se doveste spiegare le regole a chi deve scrivere il programma senza averlo mai visto giocare, cosa dovreste precisare?

## Progettiamo il gioco

Questa parte si fa alla lavagna. Compiliamo insieme le due tabelle: prima le variabili, poi le funzioni.

**Variabili**

| Nome | Cosa contiene | Tipo | Valore iniziale | Cambia durante la partita? |
|---|---|---|---|---|
| | | | | |
| | | | | |
| | | | | |
| | | | | |
| | | | | |

**Funzioni**

| Nome | Cosa riceve | Cosa restituisce | Casi particolari |
|---|---|---|---|
| | | | |
| | | | |
| | | | |
| | | | |

## Dal progetto al codice

Con l'aiuto di un LLM traduciamo in codice il progetto che abbiamo disegnato, lo leggiamo insieme e lo mettiamo alla prova.

**Tuple e liste**

In [12]:
a = (4, 6, 5, 7, 9, 2)
b = [1, 2, 3, 4, 5, 6]
holes = [(0, 1), (5, 8), (2, 3)]
print( holes[1][1] )

8


#### Griglia

In [17]:
import random

GRID_WIDTH, GRID_HEIGHT = 10, 10
NUMBER_OF_HOLES = 5

player = (0, 0)
end = (GRID_WIDTH - 1, GRID_HEIGHT - 1)

available_positions = []
for y in range(GRID_HEIGHT):
    for x in range(GRID_WIDTH):
        position = (x, y)
        if position != player and position != end:
            available_positions.append(position)
holes = random.sample(available_positions, NUMBER_OF_HOLES)

def is_valid_coordinate(x, y):
    """Le coordinate sono: 
    x per colonna (asse orizzontale)
    y per riga (asse verticale)
    La progressione è da sinistra a destra per X
    e da alto a basso per Y
    L'origine è quindi in alto a sinistra

    Args:
        x (int): coordinata colonna
        y (int): coordinata riga

    Returns:
        bool: se la cella è entro i margini della griglia
    """
    is_valid_x = x >= 0 and x < GRID_WIDTH
    is_valid_y = y >= 0 and y < GRID_HEIGHT
    return is_valid_x and is_valid_y



In [18]:
holes

[(1, 8), (7, 1), (7, 5), (8, 4), (9, 1)]